In [ ]:
10강 코드

시험이나 코딩 문제 기준으로 보면 아래 코드들을 우선적으로 외우고 이해하는 게 좋다.


① Tensor 기본
import torch

X = torch.tensor([[1.0, 4.0, 7.0],
                  [2.0, 3.0, 6.0]])

print(X.shape)   # torch.Size([2, 3])
print(X.dtype)   # torch.float32

X.mean()
X.max(dim=0)

X @ X.T
핵심
X.shape
X.dtype
X @ X.T
X.T

PyTorch Tensor는 NumPy 배열과 비슷하게 연산할 수 있고, 행렬곱은 @를 사용한다. 책에서도 Tensor의 shape, dtype, indexing, mean(), max(), 행렬곱 등을 기본으로 설명한다.


② CPU ↔ GPU

가장 기본적인 코드:

device = "cuda" if torch.cuda.is_available() else "cpu"

X = X.to(device)

모델도:

model = model.to(device)

그리고 학습할 때:

X_batch = X_batch.to(device)
y_batch = y_batch.to(device)
GPU에 바로 Tensor 만들기
X = torch.randn(10, device="cuda")

또는

X = torch.randn(10).to("cuda")


③ Autograd

10강에서 가장 중요한 부분 중 하나.

x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()
print(x.grad)

결과:
tensor(4.)

왜냐하면
y=x^2

이므로

{dy}{dx}=2x=4


④ Gradient를 사용하지 않을 때
방법 1
x = torch.tensor(2.0)

기본적으로:
requires_grad=False

방법 2
with torch.no_grad():
    y = model(X)

방법 3
y = x.detach()


⑤ In-place 연산
PyTorch에서 뒤에 _가 붙으면 in-place 연산이다.

torch.exp(x)
→ 새로운 Tensor 생성

torch.exp_(x)
→ x 자체를 수정

마찬가지로

torch.relu(x)
torch.relu_(x)

주의
Autograd를 사용하는 Tensor를 직접 수정하면 오류가 발생할 수 있다.

x = torch.tensor(2.0, requires_grad=True)
x.cos_()
이런 식의 연산은 특히 조심해야 한다.


⑥ nn.Linear
model = nn.Linear(100, 200)
의 의미:

입력 100개
   ↓
Linear
   ↓
출력 200개

내부적으로:

y=XW^T+b

따라서
model.weight.shape

→

[200, 100]
model.bias.shape

→

[200]

입력이
[32, 100]

이면 출력은
[32, 200]

이다.


⑦ Sequential로 MLP 만들기

model = nn.Sequential(
    nn.Linear(54, 200),
    nn.ReLU(),
    nn.Linear(200, 100),
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, 7)
)

구조는:

54
 ↓
200
 ↓ ReLU
100
 ↓ ReLU
50
 ↓ ReLU
7

중요한 규칙

앞 레이어의 출력 개수 = 다음 레이어의 입력 개수

nn.Linear(54, 200)
nn.Linear(200, 100)
nn.Linear(100, 50)


⑧ Loss + Optimizer

예:
criterion = nn.MSELoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

학습의 핵심
y_pred = model(X)

loss = criterion(y_pred, y)

loss.backward()

optimizer.step()

optimizer.zero_grad()

이 순서를 거의 반드시 이해해야 한다.


⑨ PyTorch Training Loop

가장 중요한 코드.

for epoch in range(n_epochs):
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

암기

데이터 → 예측 → Loss → Gradient → Update

X
↓
model(X)
↓
y_pred
↓
loss
↓
loss.backward()
↓
optimizer.step()


⑩ DataLoader
from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

그리고:

for X_batch, y_batch in train_loader:
    ...
왜 사용하는가?

전체 데이터를 한꺼번에 넣는 것이 아니라

전체 데이터
↓
Batch 1
Batch 2
Batch 3
...

형태로 나누어서 Mini-batch Gradient Descent를 수행하기 위해서다.
책에서도 DataLoader가 batch 단위로 데이터를 효율적으로 불러오고
shuffle=True로 epoch마다 데이터를 섞을 수 있다고 설명한다.


⑪ Dataset

직접 Dataset을 만들 때는 기본적으로:

class MyDataset(Dataset):

    def __len__(self):
        return ...

    def __getitem__(self, index):
        return ...

즉 핵심은:

__len__()
__getitem__()


⑫ model.train() / model.eval()

학습:

model.train()

평가:

model.eval()

그리고 평가할 때:

with torch.no_grad():
    y_pred = model(X)
왜?

Dropout이나 BatchNorm처럼 학습 모드와 평가 모드의 동작이 다른 layer가 있기 때문이다.


⑬ Classification Loss

10강에서 매우 중요.

다중 클래스
criterion = nn.CrossEntropyLoss()

예:

고양이
강아지
말
새

처럼 여러 클래스 중 하나를 선택.

중요:

nn.Linear(..., n_classes)

뒤에 Softmax를 붙이지 않는다.

model = nn.Sequential(
    nn.Linear(50, 7)
)

그리고

loss = nn.CrossEntropyLoss()

를 사용한다.

책에서도 CrossEntropyLoss는 logits를 직접 입력받으므로 output layer에 Softmax를 붙이지 않는다고 설명한다.

Binary Classification
nn.BCEWithLogitsLoss()
Multiclass
nn.CrossEntropyLoss()
NLL
nn.NLLLoss()


⑭ Custom Module

기본 형태:

class MyModel(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, X):
        ...

예를 들어:

class MyModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.linear1 = nn.Linear(10, 20)
        self.linear2 = nn.Linear(20, 1)

    def forward(self, X):
        X = F.relu(self.linear1(X))
        return self.linear2(X)
핵심
__init__
→ layer 정의

forward
→ 데이터가 layer를 통과하는 과정


⑮ nn.Parameter

직접 가중치를 만들 수도 있다.

self.weight = nn.Parameter(
    torch.randn(out_features, in_features)
)

self.bias = nn.Parameter(
    torch.zeros(out_features)
)

그리고:

z = X @ self.weight.T + self.bias

return F.relu(z)


⑯ Image Classifier 구조

Fashion MNIST 같은 이미지 분류에서는:

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 300),
    nn.ReLU(),
    nn.Linear(300, 100),
    nn.ReLU(),
    nn.Linear(100, 10)
)

중요한 것은:

[Batch, 1, 28, 28]
        ↓
Flatten
        ↓
[Batch, 784]
        ↓
Linear
        ↓
10 logits

그리고:
nn.CrossEntropyLoss()
사용.


⑰ Optuna

하이퍼파라미터를 자동으로 찾을 때:

def objective(trial):

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-2,
        1.0,
        log=True
    )

    n_layers = trial.suggest_int(
        "n_layers",
        1,
        3
    )

    ...

그리고:

study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=50
)

결과:
study.best_value
study.best_params


⑱ 모델 저장/불러오기

가중치 저장:

torch.save(
    model.state_dict(),
    "model.pth"
)

불러오기:

model.load_state_dict(
    torch.load("model.pth")
)

핵심은 state_dict()


⑲ torch.compile()

10강 후반부에서:

model = torch.compile(model)

을 사용해서 PyTorch 모델을 컴파일/최적화할 수 있다.

In [ ]:
# 10강 Exercises — PyTorch

## 1번 Tensor와 NumPy의 차이

### 문제
> PyTorch is similar to NumPy in many ways, but it offers some extra features. Can you name the most important ones?

### 번역
PyTorch는 여러 면에서 NumPy와 비슷하지만, 몇 가지 추가적인 기능을 제공합니다. **가장 중요한 추가 기능은 무엇인가?**

### 정답
PyTorch Tensor의 중요한 추가 기능은 크게 **2가지**이다.

1. **GPU 등 하드웨어 가속기를 사용할 수 있다.**
2. **자동 미분(autodifferentiation)을 지원한다.**


### 핵심
**NumPy → CPU 중심 배열 계산**
**PyTorch → Tensor + GPU + Autograd**

---

# 2번

### 문제
> What is the difference between `torch.exp()` and `torch.exp_()`, or between `torch.relu()` and `torch.relu_()`?

### 번역
`torch.exp()`와 `torch.exp_()`의 차이는 무엇인가?
또는 `torch.relu()`와 `torch.relu_()`의 차이는 무엇인가?

### 정답
**밑줄 `_`가 붙은 함수는 in-place 연산이다.**

| 함수              | 의미                         |
| --------------- | -------------------------- |
| `torch.exp()`   | 새로운 Tensor를 만들어 결과 반환      |
| `torch.exp_()`  | 기존 Tensor 자체를 변경           |
| `torch.relu()`  | 새로운 Tensor를 만들어 ReLU 결과 반환 |
| `torch.relu_()` | 기존 Tensor 자체를 ReLU 결과로 변경  |

예를 들어,

```python
x = torch.tensor([-1., 2.])

y = torch.relu(x)
```

이면
```text
x = [-1, 2]
y = [0, 2]
```

하지만
```python
x = torch.tensor([-1., 2.])
x.relu_()
```

이면
```text
x = [0, 2]
```

---

# 3번 GPU에 Tensor를 만드는 두 가지 방법

### 문제
> What are two ways to create a new tensor on the GPU?

### 번역
GPU에 새로운 Tensor를 만드는 방법 두 가지는 무엇인가?

### 정답

### 방법 1. CPU에서 만든 뒤 GPU로 이동

```python
M = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])

M = M.to("cuda")
```

### 방법 2. 처음부터 GPU에 생성

```python
M = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]], device="cuda")
```

책에서도 이 두 방법을 제시한다.

### 핵심 ⭐

```python
tensor.to("cuda")
```

또는

```python
tensor(..., device="cuda")
```

---

# 4번 Autograd 없이 Tensor 계산하는 세 가지 방법

### 문제
> What are three ways to perform tensor computations without using autograd?

### 번역
Autograd를 사용하지 않고 Tensor 연산을 수행하는 세 가지 방법은 무엇인가?

### 정답

대표적으로 다음 세 가지 방법이다.

### ① `torch.no_grad()`
```python
with torch.no_grad():
    y = model(x)
```

### ② `torch.inference_mode()`
```python
with torch.inference_mode():
    y = model(x)
```

### ③ Tensor의 `requires_grad`를 False로 설정
```python
x = torch.tensor(2.0, requires_grad=False)
```

또는 계산에 필요한 Tensor를 gradient tracking 없이 사용하는 방법이다.

### 핵심

학습이 아닌 **추론/평가 단계에서는 gradient 계산을 하지 않는 것**이 중요하다.

---

# 5번 In-place 연산과 Autograd

이 문제는 **10강에서 꽤 중요하다.**

### 문제 1

다음 코드가 `RuntimeError`를 발생시키는가?

```python
t = torch.tensor(2.0, requires_grad=True)
z = t.cos().exp_()
z.backward()
```

그리고 다음으로 바꾸면?

```python
z = t.cos_().exp()
```

또는

```python
z = t.exp().cos_()
```

---

## 정답

### 첫 번째

```python
t = torch.tensor(2.0, requires_grad=True)
z = t.cos().exp_()
z.backward()
```

**→ RuntimeError 발생**

이유는 `exp_()`가 **in-place 연산**인데, `t.cos()`의 결과가 backward를 위해 필요하기 때문이다.

---

### 두 번째

```python
z = t.cos_().exp()
```

**→ RuntimeError 발생**

이번에는 `t` 자체를 `cos_()`가 변경한다.

`t`는 `requires_grad=True`인 **leaf Tensor**이기 때문에 이런 in-place 변경이 허용되지 않는다.

---

### 세 번째

```python
z = t.exp().cos_()
```

**→ RuntimeError 발생**

`exp()` 결과에 `cos_()`를 적용하는 것이지만, backward 과정에서 필요한 값을 in-place로 덮어쓸 수 있기 때문에 문제가 발생한다.

### 핵심

> **Autograd가 backward에 필요한 값을 저장하고 있는데 그것을 in-place로 바꾸면 문제가 발생할 수 있다.**

---

## 두 번째 코드

```python
u = torch.tensor(2.0, requires_grad=True)
v = u + 1
w = v.cos() * v.sin_()
w.backward()
```

### 정답

이 역시 **RuntimeError가 발생한다.**

`v`에 대해 `sin_()`가 in-place 연산을 수행하면서 `v`의 기존 값이 변경되기 때문이다.

---

### `v.cos_()`로 바꾸면?

```python
w = v.cos_() * v.sin()
```

이 경우에도 **RuntimeError가 발생할 수 있다.**

그리고 중요한 점:

> **두 경우의 `w` 값은 원칙적으로 같은 수학적 식을 계산하지만, in-place 연산 때문에 autograd의 계산 그래프가 손상될 수 있다.**

---

# 6번 `Linear(100, 200)`

### 문제
> Suppose you create a `Linear(100, 200)` module. How many neurons does it have? What is the shape of its weight and bias parameters? What input shape does it expect? What output shape does it produce?

### 번역
`Linear(100, 200)` 모듈을 만들었다고 하자.

1. 몇 개의 뉴런을 가지는가?
2. Weight의 shape은?
3. Bias의 shape은?
4. 어떤 입력 shape을 기대하는가?
5. 출력 shape은 무엇인가?

---

## 정답

```python
nn.Linear(100, 200)
```

의 의미는

> **입력 뉴런 100개 → 출력 뉴런 200개**

### ① 뉴런 수

**200개**

Linear layer에서 `out_features = 200`이므로 출력 뉴런이 200개이다.

### ② Weight

```text
(200, 100)
```

즉,
W = {R}^{200\times100}


### ③ Bias

```text
(200,)
```

### ④ 입력

일반적인 batch 입력은

```text
(batch_size, 100)
```

### ⑤ 출력

```text
(batch_size, 200)
```

### 정리

```text
nn.Linear(100, 200)

입력: 100
       ↓
   [Linear]
       ↓
출력: 200
```

| 항목     | Shape               |
| ------ | ------------------- |
| Input  | `(batch_size, 100)` |
| Weight | `(200, 100)`        |
| Bias   | `(200,)`            |
| Output | `(batch_size, 200)` |

---

# 7번 PyTorch Training Loop

### 문제
> What are the main steps of a PyTorch training loop?

### 번역
PyTorch의 **학습 루프(training loop)의 주요 단계**는 무엇인가?

### 정답
일반적인 순서는 다음과 같다.

### ① Batch 가져오기

```python
for X_batch, y_batch in train_loader:
```

### ② GPU로 이동

```python
X_batch = X_batch.to(device)
y_batch = y_batch.to(device)
```

### ③ Forward propagation

```python
y_pred = model(X_batch)
```

### ④ Loss 계산

```python
loss = criterion(y_pred, y_batch)
```

### ⑤ Gradient 초기화

```python
optimizer.zero_grad()
```

### ⑥ Backpropagation

```python
loss.backward()
```

### ⑦ Parameter 업데이트

```python
optimizer.step()
```

### 전체 순서

```text
Batch
 ↓
Forward
 ↓
Loss 계산
 ↓
Gradient 초기화
 ↓
Backward
 ↓
Optimizer Step
 ↓
다음 Batch
```

시험에서는 이 순서를 기억하는 게 중요하다.

---

# 8번 왜 Model을 GPU로 옮긴 후 Optimizer를 만들어야 하는가?

### 문제
> Why is it recommended to create the optimizer after the model is moved to the GPU?

### 번역
왜 Model을 GPU로 이동시킨 **후에** Optimizer를 만드는 것이 권장되는가?

### 정답
Optimizer는 **Model의 parameter들을 참조하여 관리**하기 때문이다.

따라서

```python
model.to(device)
optimizer = torch.optim.Adam(model.parameters())
```

순서로 만드는 것이 안전하다.

반대로

```python
optimizer = torch.optim.Adam(model.parameters())
model.to(device)
```

처럼 하면 Optimizer가 CPU의 parameter를 참조하는 상황이 생길 수 있다.

### 핵심

> **Model을 먼저 GPU로 이동 → 그다음 Optimizer 생성**

```python
model.to(device)
optimizer = Adam(model.parameters())
```

---

# 9번 GPU 학습을 빠르게 만드는 DataLoader 옵션

### 문제
> What DataLoader options should you generally set to speed up training when using a GPU?

### 번역
GPU를 사용할 때 학습 속도를 높이기 위해 일반적으로 어떤 `DataLoader` 옵션을 설정해야 하는가?

### 정답
주요 옵션은 다음과 같다.

```python
DataLoader(
    ...,
    pin_memory=True,
    num_workers=...
)
```

필요하면

```python
prefetch_factor=...
persistent_workers=True
```

도 사용할 수 있다.

### ① `pin_memory=True`
CPU 메모리를 page-locked memory로 만들어 **GPU로 데이터를 더 빠르게 전송**할 수 있다.

### ② `num_workers`
여러 프로세스를 이용하여 다음 batch를 미리 준비한다.

### ③ `prefetch_factor`
각 worker가 미리 준비해놓는 batch의 수를 조절한다.

### ④ `persistent_workers=True`
epoch가 바뀌어도 worker를 유지한다.

책에서도 GPU 학습 시 `pin_memory=True`, `num_workers` 등을 사용하여 데이터 전송과 preprocessing을 병렬화할 수 있다고 설명한다.

---

# 10번 PyTorch의 Classification Loss

### 문제
> What are the main classification losses provided by PyTorch, and when should you use each of them?

### 번역
PyTorch에서 제공하는 주요 **분류 손실 함수(classification loss)**&#xB294; 무엇이며, 각각 언제 사용하는가?

### 정답
대표적으로 다음 세 가지가 중요하다.

| Loss                | 사용 상황                    |
| ------------------- | ------------------------ |
| `CrossEntropyLoss`  | 다중 클래스 분류                |
| `BCEWithLogitsLoss` | 이진 분류 / Multi-label      |
| `BCELoss`           | Sigmoid 출력값을 직접 넣는 이진 분류 |

---

## ① `CrossEntropyLoss`

가장 대표적인 **multi-class classification** loss.

예:
```text
고양이 / 개 / 말 / 새
```

처럼 하나의 입력이 **하나의 클래스**에 해당하는 경우.

보통 모델의 마지막 layer에서 **softmax를 직접 적용하지 않고 logits를 그대로** 넣는다.

```python
loss = nn.CrossEntropyLoss()
```

---

## ② `BCEWithLogitsLoss`

Binary classification 또는 multi-label classification.

예:

```text
Spam인가? → 0/1
```

또는

```text
Spam?   → 1
Urgent? → 1
```

처럼 여러 label이 동시에 1이 될 수 있는 경우.

`Sigmoid + BCELoss`를 합쳐놓은 형태라서 일반적으로 이것을 사용하는 것이 좋다.

---

## ③ `BCELoss`

Sigmoid를 **이미 적용한 확률값**을 입력으로 사용할 때.

```python
y_pred = torch.sigmoid(logits)
loss = nn.BCELoss()(y_pred, y)
```

하지만 일반적으로는

```python
nn.BCEWithLogitsLoss()
```

를 사용하는 것이 더 편리하고 수치적으로 안정적이다.

---

# 11번 `model.train()`과 `model.eval()`

### 문제
> Why is it important to call `model.train()` before training and `model.eval()` before evaluation?

### 번역
학습 전에 `model.train()`을 호출하고 평가 전에 `model.eval()`을 호출하는 것이 왜 중요한가?

### 정답
**모델의 일부 layer가 training mode와 evaluation mode에서 다르게 동작하기 때문이다.**

대표적으로

* Dropout
* Batch Normalization

이 있다.

### 학습

```python
model.train()
```

→ Dropout 활성화
→ BatchNorm이 batch statistics 사용

### 평가

```python
model.eval()
```

→ Dropout 비활성화
→ BatchNorm이 학습된 statistics 사용

### 핵심

```text
학습 → model.train()

평가 → model.eval()
```

---

# 12번 `torch.jit.trace()` vs `torch.jit.script()`

### 문제
> What is the difference between `torch.jit.trace()` and `torch.jit.script()`?

### 번역
`torch.jit.trace()`와 `torch.jit.script()`의 차이는 무엇인가?

### 정답

### `torch.jit.trace()`

실제 입력 데이터를 모델에 넣어서 **실행되는 연산을 추적(trace)**&#xD55C;다.

```python
traced_model = torch.jit.trace(model, example_input)
```

따라서 입력에 따라 실행 경로가 달라지는 **동적 제어 흐름**을 제대로 처리하지 못할 수 있다.

---

### `torch.jit.script()`

Python 코드를 분석하여 **모델의 계산 구조를 scripting**한다.

따라서 `if`, `for` 같은 제어 흐름도 표현할 수 있다.

### 핵심

|            | trace     | script    |
| ---------- | --------- | --------- |
| 방법         | 실제 실행을 추적 | 코드 분석     |
| 동적 제어 흐름   | 약함        | 지원        |
| 입력 example | 필요        | 일반적으로 불필요 |

---

# 13번 Autograd로 Gradient 구하기

### 문제
Autograd를 사용하여 다음 함수의 gradient vector를 구하라.

$$
f(x,y)=\sin\left(\frac{x^2}{y}\right)
$$

단,

$$
(x,y)=(1.2,3.4)
$$

---

## 정답

Gradient는

$$
\nabla f=
\begin{bmatrix}
\frac{\partial f}{\partial x}\\
\frac{\partial f}{\partial y}
\end{bmatrix}
$$

---

# 14번 Custom Dense Module 만들기

### 문제
> Create a custom Dense module that replicates the functionality of an `nn.Linear` module followed by an `nn.ReLU` module.

> Try implementing it first using the `nn.Linear` and `nn.ReLU` modules, and then reimplement it using `nn.Parameter` and the `relu()` function.

### 번역
`nn.Linear` 모듈 다음에 `nn.ReLU` 모듈을 연결한 것과 **동일한 기능을 수행하는 Custom Dense 모듈**을 만들어라.

먼저 `nn.Linear`와 `nn.ReLU` 모듈을 사용해서 구현하고, 그다음에는 `nn.Parameter`와 `relu()` 함수를 사용하여 다시 구현하라.

---

## 정답 ① `nn.Linear` + `nn.ReLU`

```python
import torch
import torch.nn as nn

class Dense(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()

        self.linear = nn.Linear(input_size, output_size)
        self.relu = nn.ReLU()

    def forward(self, X):
        return self.relu(self.linear(X))
```

사용:

```python
dense = Dense(100, 200)
```

즉,

```text
X
 ↓
Linear
 ↓
ReLU
 ↓
Output
```

---

## 정답 ② `nn.Parameter` + `relu()`

이번에는 Linear를 직접 구현한다.

```python
class Dense(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()

        self.weights = nn.Parameter(
            torch.randn(output_size, input_size)
        )
        self.bias = nn.Parameter(
            torch.zeros(output_size)
        )

    def forward(self, X):
        return torch.relu(X @ self.weights.T + self.bias)
```

핵심은
Y = XW^T+b

그 다음
Output=ReLU(Y)

---

# 15번 CoverType Classification MLP

### 문제
> Build and train a classification MLP on the CoverType dataset.

### 번역
**CoverType 데이터셋을 사용하여 분류용 MLP를 구축하고 학습하라.**

세부적으로 다음을 수행한다.

### a.

`sklearn.datasets.fetch_covtype()`를 사용하여 데이터셋을 불러오고, 이 데이터를 위한 **Custom PyTorch Dataset**을 만들어라.

### b.

Training / Validation / Testing을 위한 **DataLoader**를 만들어라.

### c.

이 분류 문제를 해결하기 위한 **Custom MLP module**을 만들어라.

이전 문제에서 만든 Custom Dense module을 선택적으로 사용할 수 있다.

### d.

GPU에서 모델을 학습하고 **Test accuracy 93%**에 도달하도록 시도하라.

---

# 15-a 정답

CoverType 데이터 로드:

```python
from sklearn.datasets import fetch_covtype

covtype = fetch_covtype()

X = covtype.data
y = covtype.target
```

PyTorch Dataset:

```python
import torch
from torch.utils.data import Dataset

class CoverTypeDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y - 1, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]
```

여기서

```python
y - 1
```

을 하는 이유는 CoverType의 class label이 1부터 시작하기 때문이다.

PyTorch classification에서 사용할 class index는

```text
0, 1, 2, ...
```

형태로 맞추는 것이 편하다.

---

# 15-b 정답 — DataLoader

먼저 train / validation / test로 분리한다.

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)
```

Dataset:

```python
train_data = CoverTypeDataset(X_train, y_train)
valid_data = CoverTypeDataset(X_valid, y_valid)
test_data = CoverTypeDataset(X_test, y_test)
```

DataLoader:

```python
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_data,
    batch_size=256,
    shuffle=True,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_data,
    batch_size=256,
    pin_memory=True
)

test_loader = DataLoader(
    test_data,
    batch_size=256,
    pin_memory=True
)
```

---

# 15-c 정답 — MLP

CoverType는 입력 feature가 **54개**, class가 **7개**이다.

따라서 기본적인 MLP는

```text
54
 ↓
Hidden Layer
 ↓
Hidden Layer
 ↓
7
```

구조로 만들 수 있다.

예:

```python
import torch.nn as nn

class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(54, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 7)
        )

    def forward(self, X):
        return self.model(X)
```

### 출력층에 Softmax를 넣지 않는 것이 중요 ⭐

Loss를

```python
nn.CrossEntropyLoss()
```

로 사용할 경우 마지막 layer에서 **Softmax를 직접 넣지 않는다.**

즉,

```text
Linear(64 → 7)
       ↓
    logits
       ↓
CrossEntropyLoss
```

---

# 15-d 정답 — GPU에서 학습

Device:

```python
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
```

Model:

```python
model = MLP().to(device)
```

Optimizer:

```python
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
```

Loss:

```python
criterion = nn.CrossEntropyLoss()
```

학습:

```python
for epoch in range(20):

    model.train()

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        y_pred = model(X_batch)

        loss = criterion(y_pred, y_batch)

        loss.backward()

        optimizer.step()
```

평가할 때는

```python
model.eval()
```

을 사용한다.